# Let's Start by Exploring Acknowledgment Sections

👋 Hello and welcome!  
In this notebook, we begin our journey into the world of **grant information extraction** by exploring how **acknowledgments** are written in scientific publications.

This is the first step of our exploratory project, developed in the context of Carolina’s **Torres Quevedo** grant and connected to our work on the **SciLake** project. Ultimately, we aim to uncover patterns in how funding sources, grant numbers, and institutional support are expressed — with the broader goal of building structured knowledge about research funding.

---

## 📚 Data Sources

In this initial notebook, we’ll gather and explore acknowledgment sections from two main sources:

1. **PubMed**  
   We'll retrieve acknowledgment texts from open-access PubMed articles using their APIs and metadata.

2. **SciLake Dataset**  
   We’ll also use full-text data collected in the context of the SciLake project. These papers span multiple domains and include rich textual content from which acknowledgments can be extracted.

---

## 🎯 Goals of This Notebook

- Collect a sample of acknowledgment sections from both sources.
- Do a first-pass qualitative exploration:
  - How are funders mentioned?
  - How are grant numbers formatted?
  - Are there domain-specific differences?
- Identify common patterns and challenges that might influence our extraction strategy.

---

1. **Setup & Imports**  
   Basic dependencies and helper functions for text processing and data loading.


In [1]:
# Import necessary libraries
import sys
import os
import importlib
from tqdm.notebook import tqdm
from datasets import load_dataset

# Add the 'src' directory to the Python path so we can import from it
sys.path.append(os.path.abspath('../src'))

<module 'data_gathering' from '/home/siris/Repositories/exploring-grant-information/src/data_gathering.py'>

2. **Collect Acknowledgments from PubMed**  
   Using PubMed’s API or pre-downloaded data to extract acknowledgment sections.

In this part we are extracting acknowledgment sections from **biomedical publications** using PubMed. PubMed, a key resource in life sciences and biomedical research, provides access to a vast database of articles often including acknowledgments that list funding sources, contributors, and collaborations. Using PubMed's API or pre-downloaded data, we extract these acknowledgment sections to analyze patterns in research funding, collaborations, and the roles of various contributors, which are valuable for understanding trends in biomedical research and grant distribution.

In [2]:
# 1. Import the module from 'src/' (e.g., data_gathering)
import data_gathering
importlib.reload(data_gathering)

# Now we can import functions from src
from data_gathering import get_pmc_ids, select_random_ids, fetch_articles, parse_acknowledgments

# Main script execution
pmc_ids = get_pmc_ids()
random_ids = select_random_ids(pmc_ids, num_ids = 100)
articles_xml = fetch_articles(random_ids)
acknolwedgment_sections = parse_acknowledgments(articles_xml)

In [4]:
# Display some of the acknowledgments in the list
acknolwedgment_sections[:3]

['Special thanks are extended to Subhrangsu Mukherjee\nfor sharing his feedback for the improvement of this manuscript. The\nauthor thanks Weigang Zhu, Tobin Marks, and the rest of the collaborative\nauthor team involved in our experimental PM6:Y6 publication. 59',
 'The research was funded by grants from the Spanish Government (Transfer, MCIN/ AEI/ CGL2016–80124-C2-1-P) and Biodivrestore Cofund 2020 (FishMe, PCI2022-132949). E.F. acknowledges her predoctoral fellowship (BES-2017-081553) and A.B. additional funding through the national program P1-0255 of the Slovenian Ministry of Science and Education (ARRS). The authors thank Lluís Camarero, Sergi Pla-Rabès, and Meritxell Batalla for support at different stages of the study; the Servei d’Anàlisi Química (SAQ) from the Universitat Autònoma de Barcelona for chemical analysis advice, and HPLC facilities for pigment analyses; and AllGenetics & Biology SL (A Coruña, Spain) for sequencing.',
 'The authors sincerely express their gratitude t

3. **Load Acknowledgments from SciLake Dataset**  
   Parsing full-text files and isolating the acknowledgment content.

Here we load acknowledgment sections from the SciLake dataset, which includes full-text files across multiple domains. We parse these files to isolate and extract the acknowledgment content, providing a diverse sample of research publications. This dataset covers a range of domain-specific sections, including Neuroscience 🧠, Cancer 🦀, Transport 🛻, and Energy 🪫, allowing us to explore acknowledgment patterns across different fields of study and gain insights into funding sources, research collaborations, and contributors within these specific domains.

In [22]:
import re
import random
from datasets import load_dataset

# Download dataset
dataset_ds = load_dataset("SIRIS-Lab/scilake-fulltext-corpus")

# Define a list of keywords/phrases to match for different variations of acknowledgment and funding sections
keywords = [
    r'\bAcknowledg\w*\b',  # Acknowledgment, Acknowledgements
    #r'\bFund\w*\b',        # Funding, Funded
    #r'\bGrant\w*\b'        # Grant, Grants
]

# This will store the section contents for all splits
sections_sample = []

# Loop through all splits (train, test, validation)
for split in dataset_ds.keys():
    print(f"Processing split: {split}")

    # Loop through the 'fulltext_additional' section of each split
    for sections in dataset_ds[split]['fulltext_additional']:
        filtered_sections = [
            section for section in sections 
            if section['section_name'] and any(re.search(keyword, section['section_name'], re.IGNORECASE) for keyword in keywords)
        ]

        # Append the section contents
        for section in filtered_sections:
            if section['section_content'] != '':
                sections_sample.append(section['section_content'])

# All sections collected from all splits
all_sections = list(sections_sample)

Processing split: cancer
Processing split: transport
Processing split: neuroscience
Processing split: energy
Processing split: general


In [26]:
all_sections[:10]

['We thank the patients who participated in the study and their supportive families, as well as the investigators and clinical research staff from the study centers.Editorial support was provided by Melanie Sweetlove, and funded by Pharmacyclics LLC, an AbbVie Company. ',
 'Not applicable. ',
 'We are grateful to C Cottonham, S Hankeova, and G Hernandez for their helpful discussions.We thank the Genentech Research Pathology, Necropsy, and Histology laboratories for their experimental contributions.We appreciate the insightful feedback and comments on the paper from L Mosteiro, E Reyes, and B Biehs. ',
 'This work was supported by a National Institutes of Health grants 1R01HL114823 (SCB), 1P30GM103342 (RAN), 8P20GM103444-07 (RAN), R01HL127692 (RAN, DM), American Heart Association 15GRNT25080052 (RAN). ',
 'Funding: This research was funded by the Austrian National Bank, "Jubilaeumsfonds"-Grant No. 13012; by the "Initiative Krebsforschung", UE71104017, UE1504001, and UE711043037; by a Cl

4. **Initial Exploration**  
   Basic statistics (e.g., average length, number of entities, language patterns).  
   Display a few examples from both datasets.

5. **Pattern Spotting & Annotation Ideas**  
   Jot down any emerging ideas for tagging funders, grants, or institutions.


6. **Wrap-up & Next Steps**  
   Notes on what’s interesting, what’s messy, and how we could continue.
